In [1]:

VERSION = 2

import pandas as pd
import numpy as np
import lightgbm as lgb


from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, LassoCV, ElasticNetCV
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupShuffleSplit, GroupKFold
from scipy.stats import pearsonr, spearmanr


In [2]:


df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\Updated02\community_index_v9.csv")
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Loaded: 48,000 rows × 168 columns


In [3]:

r0_cols     = [f"R0_c{k}"     for k in range(1, 21)]   # 20 concentration cols
active_cols = [f"active_c{k}" for k in range(1, 21)]   # 20 binary flag cols


if VERSION == 1:
    # Full model: species membership + all nutrient info + structural
    species_cols = [f"Species_{i:04d}" for i in range(2, 102)]  # 100 binary cols
    feature_cols = species_cols + r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Full (species + nutrients + size)"

elif VERSION == 2:
    # Nutrient + size: drop species membership, keep community_size
    feature_cols = r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Nutrients + size (no species)"

elif VERSION == 3:
    # Pure nutrient: only raw chemistry — no species, no community_size
    # diversity_level dropped too: it is redundant since XGBoost/LightGBM
    # can infer it directly by counting non-zero R0 columns.
    feature_cols = r0_cols + active_cols
    version_label = "Pure nutrients only (no species, no size)"

else:
    raise ValueError("VERSION must be 1, 2, or 3")

print(f"\nVersion: {version_label}")
print(f"Features: {len(feature_cols)}")


missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"{len(missing)} columns not found in file. First 3: {missing[:3]}\n"
                   f"Make sure you are loading community_index_v8_backup.csv (30,000 rows), "
                   f"not community_dataset_v8_backup.csv (792,000 rows).")

X = df[feature_cols]
y = df['community_suppression']
print(f"X: {X.shape}   y range: [{y.min():.3f}, {y.max():.3f}]")


Version: Nutrients + size (no species)
Features: 42
X: (48000, 42)   y range: [0.000, 0.999]


In [4]:
strat_key = df['community_size'].astype(str) + "_" + df['diversity_level'].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)


groups = df['broth_id'].values # used later for groupkfold

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test set:  {X_test.shape[0]:,} rows")
print(f"Unique broths: {df['broth_id'].nunique()}")
print(f"\nDiversity level distribution in test set:")
print(df.loc[X_test.index, 'diversity_level'].value_counts().sort_index())

Train set: 38,400 rows
Test set:  9,600 rows
Unique broths: 32

Diversity level distribution in test set:
diversity_level
1     3000
2     2400
4     1800
8     1200
16    1200
Name: count, dtype: int64


In [5]:
# lgbm initial fit

early_stop = lgb.early_stopping(stopping_rounds=30, verbose=False)
# lgb.early_stopping: stops training if validation metric hasn't improved
# for `stopping_rounds` consecutive rounds. Returns a callback object.

log_eval = lgb.log_evaluation(period=0)
# lgb.log_evaluation(period=0): suppresses all per-round console output.
# Set period=50 if you want to see progress every 50 rounds.

model_init = lgb.LGBMRegressor(
    n_estimators=500,
    num_leaves=63,

    max_depth=-1,
    learning_rate=0.05,
    subsample=0.8,
    bagging_freq=1,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# LightGBM eval_set syntax is the same as XGBoost.
model_init.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[early_stop, log_eval]
)

yp_init  = model_init.predict(X_test)
r2_init  = r2_score(y_test, yp_init)
mae_init = mean_absolute_error(y_test, yp_init)
print(f"LightGBM initial:  R²={r2_init:.4f}  MAE={mae_init:.4f}  "
      f"best_iter={model_init.best_iteration_}")

LightGBM initial:  R²=0.8624  MAE=0.0785  best_iter=151


In [6]:
# hyperparameter tuning

param_grid = {                               # best hyperparams from first run used
    'num_leaves':      [31],
    # The primary complexity knob in LightGBM.
    # 31 = shallow/conservative, 127 = deep/expressive.

    'learning_rate':   [0.05],

    'subsample':       [1.0],
    # Note: subsample only works when bagging_freq > 0 (set in base_est below).

    'colsample_bytree':[0.6],
}


cv = KFold(n_splits=3, shuffle=True, random_state=42)

base_est = lgb.LGBMRegressor(
    n_estimators=200,
    max_depth=-1,
    bagging_freq=1,
    min_child_samples=20,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

grid_search = GridSearchCV(
    estimator=base_est,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)
grid_search.fit(X_train, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Fitting 3 folds for each of 1 candidates, totalling 3 fits

Best params: {'colsample_bytree': 0.6, 'learning_rate': 0.05, 'num_leaves': 31, 'subsample': 1.0}
Best CV RMSE: 0.1404


In [15]:
# final model

model_final = lgb.LGBMRegressor(
    **grid_search.best_params_,
    n_estimators=1000,
    max_depth=-1,
    bagging_freq=1,
    min_child_samples=20,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

evals_log         = {}
early_stop_final  = lgb.early_stopping(stopping_rounds=40, verbose=False)
log_eval_final    = lgb.log_evaluation(period=50)
record_eval_final = lgb.record_evaluation(evals_log)

model_final.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    eval_names=['train', 'valid'],
    callbacks=[early_stop_final, log_eval_final, record_eval_final]
)

# After fit:
tr_rmse  = np.sqrt(np.array(evals_log['train']['l2']))
val_rmse = np.sqrt(np.array(evals_log['valid']['l2']))
rounds   = range(1, len(tr_rmse) + 1)
best_it  = model_final.best_iteration_       # note the underscore -- LightGBM uses best_iteration_

fig_tc, ax_tc = plt.subplots(figsize=(9, 4))
ax_tc.plot(rounds, tr_rmse,  color='steelblue',  lw=1.5, label='Train RMSE')
ax_tc.plot(rounds, val_rmse, color='darkorange', lw=1.5, label='Validation RMSE')
ax_tc.axvline(best_it, color='red', linestyle='--', lw=1.2,
              label=f'Best iteration = {best_it}')
ax_tc.set_xlabel('Boosting round')
ax_tc.set_ylabel('RMSE')
ax_tc.set_title(f'LightGBM Training Curve  --  best iter={best_it},  val RMSE={val_rmse[best_it - 1]:.4f}')
ax_tc.legend(fontsize=9)
ax_tc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lgbm_training_curve_0202.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: lgbm_training_curve_0202.png")

yp_final   = model_final.predict(X_test)
r2_final   = r2_score(y_test, yp_final)
mae_final  = mean_absolute_error(y_test, yp_final)
rmse_final = np.sqrt(mean_squared_error(y_test, yp_final))

print(f"\nLightGBM final:  R²={r2_final:.4f}  MAE={mae_final:.4f}  RMSE={rmse_final:.4f}")
print(f"Trees used: {model_final.best_iteration_}")

[50]	train's l2: 0.0209353	valid's l2: 0.0211277
[100]	train's l2: 0.0195247	valid's l2: 0.0197545
[150]	train's l2: 0.0194834	valid's l2: 0.0197198
[200]	train's l2: 0.0194801	valid's l2: 0.0197194
Saved: lgbm_training_curve_0202.png

LightGBM final:  R²=0.8624  MAE=0.0787  RMSE=0.1404
Trees used: 178


In [8]:
# groupkfold

gkf          = GroupKFold(n_splits=5)
fold_r2, fold_mae, fold_rmse = [], [], []

print("\nGroupKFold evaluation (held-out broths):")


for fold_num, (tr_idx, te_idx) in enumerate(
        gkf.split(X, y, groups=groups), start=1):

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    held_broths = sorted(set(df.iloc[te_idx]['broth_id']))
    held_divs   = sorted(df.iloc[te_idx]['diversity_level'].unique())
    print(f"\nFold {fold_num}: broths={held_broths}  divs={held_divs}  n_test={len(X_te):,}")

    m_fold = lgb.LGBMRegressor(
        **grid_search.best_params_,
        n_estimators=1000, min_child_samples=20, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbose=-1
    )
    m_fold.fit(
        X_tr, y_tr,
        eval_set=[(X_te, y_te)],
        callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(0)]
    )

    yp_fold = m_fold.predict(X_te)
    r2_f    = r2_score(y_te, yp_fold)
    mae_f   = mean_absolute_error(y_te, yp_fold)
    rmse_f  = np.sqrt(mean_squared_error(y_te, yp_fold))

    fold_r2.append(r2_f); fold_mae.append(mae_f); fold_rmse.append(rmse_f)
    print(f"  R²={r2_f:.4f}  MAE={mae_f:.4f}  RMSE={rmse_f:.4f}")


print(f"\nGroupKFold summary across {gkf.n_splits} folds:")
print(f"  R²:   mean={np.mean(fold_r2):.4f}  std={np.std(fold_r2):.4f}")
print(f"  MAE:  mean={np.mean(fold_mae):.4f}  std={np.std(fold_mae):.4f}")
print(f"  RMSE: mean={np.mean(fold_rmse):.4f}  std={np.std(fold_rmse):.4f}")
print(f"\nCompare to stratified-split R²={r2_final:.4f}")
print(f"Drop: {r2_final - np.mean(fold_r2):.4f}  (how much was broth memorisation)")


GroupKFold evaluation (held-out broths):

Fold 1: broths=['Broth_02', 'Broth_07', 'Broth_12', 'Broth_17', 'Broth_22', 'Broth_27', 'Broth_32']  divs=[np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16)]  n_test=10,500
  R²=0.0292  MAE=0.3494  RMSE=0.4103

Fold 2: broths=['Broth_01', 'Broth_06', 'Broth_11', 'Broth_16', 'Broth_21', 'Broth_26', 'Broth_31']  divs=[np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16)]  n_test=10,500
  R²=-0.2022  MAE=0.2074  RMSE=0.2406

Fold 3: broths=['Broth_05', 'Broth_10', 'Broth_15', 'Broth_20', 'Broth_25', 'Broth_30']  divs=[np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16)]  n_test=9,000
  R²=0.0438  MAE=0.2638  RMSE=0.3447

Fold 4: broths=['Broth_04', 'Broth_09', 'Broth_14', 'Broth_19', 'Broth_24', 'Broth_29']  divs=[np.int64(1), np.int64(2), np.int64(4), np.int64(16)]  n_test=9,000
  R²=0.2016  MAE=0.2331  RMSE=0.3200

Fold 5: broths=['Broth_03', 'Broth_08', 'Broth_13', 'Broth_18', 'Broth_23', 'Broth_28']  di

In [9]:
print("Performance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

def condition_metrics(y_true_arr, y_pred_arr):
    y_t      = np.asarray(y_true_arr)
    y_p      = np.asarray(y_pred_arr)
    r2       = r2_score(y_t, y_p)
    mae      = mean_absolute_error(y_t, y_p)
    rmse     = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _    = pearsonr(y_t, y_p)
    sr, _    = spearmanr(y_t, y_p)
    bias     = float(np.mean(y_p - y_t))
    std_true = float(np.std(y_t))
    return {'n': len(y_t), 'r2': r2, 'mae': mae, 'rmse': rmse,
            'pearson_r': pr, 'spearman_r': sr,
            'bias': bias, 'std_true': std_true}

hdr = f"{'':6} {'R²':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} {'Bias':>8} {'std(y)':>7} {'N':>6}"
print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m    = condition_metrics(grp['y_true'], grp['y_pred'])
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp['y_true'], grp['y_pred'])
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}")

Performance by diversity level and community size

By diversity level:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
D=1     0.8941  0.0716  0.1459   0.9456    0.8699   0.0043  0.4483  3,000   
D=2     0.8784  0.0756  0.1438   0.9372    0.9439   0.0001  0.4123  2,400   
D=4     0.3566  0.0797  0.1409   0.6008    0.7569   0.0084  0.1757  1,800   
D=8     0.6216  0.0827  0.1238   0.7895    0.8659  -0.0056  0.2012  1,200   
D=16    0.4363  0.0970  0.1345   0.6606    0.6910   0.0011  0.1792  1,200   
  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.

By community size:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
N=12    0.6923  0.1393  0.2066   0.8323    0.8149   0.0068  0.3724  1,920
N=16    0.7797  0.1113  0.1729   0.8830    0.8395   0.0002  0.3684  1,920
N=24 

In [10]:
# shap analysis

print('shap analysis')

explainer   = shap.TreeExplainer(model_final)
rng         = np.random.default_rng(42)
idx         = rng.choice(len(X_test), size=2000, replace=False)
shap_values = explainer.shap_values(X_test.iloc[idx])
mean_shap   = pd.Series(np.abs(shap_values).mean(0), index=feature_cols).sort_values(ascending=False)

print(f"\nTop 15 features by mean |SHAP|:")
print(mean_shap.head(15).round(5).to_string())

shap analysis

Top 15 features by mean |SHAP|:
R0_c4              0.08597
community_size     0.05980
diversity_level    0.03790
R0_c11             0.03304
R0_c1              0.03130
R0_c17             0.02926
active_c4          0.01923
active_c17         0.01766
active_c1          0.01716
active_c11         0.01446
R0_c3              0.01396
R0_c7              0.01206
R0_c16             0.01185
R0_c9              0.01042
active_c3          0.00886


In [13]:
# plots

from matplotlib.patches import Patch

output_name = f"lgbm_v{VERSION}_results_02.png"
# Output filename automatically reflects the version so you won't overwrite results.

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'LightGBM — {version_label} — MiCRM v8\n'
    f'R²={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}',
    fontsize=12, fontweight='bold', y=0.99
)

ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1,2,4,8,16], ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3']):
    mask = results_df['diversity_level'].values == d
    ax1.scatter(y_test.values[mask], yp_final[mask], alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0,1],[0,1],'r--',lw=1.5)
ax1.set_xlabel('True suppression'); ax1.set_ylabel('Predicted')
ax1.set_title(f'Predicted vs Actual  R²={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

ax2 = fig.add_subplot(2, 3, 2)
resid = y_test.values - yp_final
for n, col in zip([12,16,24,32,48], ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']):
    mask = results_df['community_size'].values == n
    ax2.scatter(yp_final[mask], resid[mask], alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Residual')
ax2.set_title('Residuals by Community Size')
ax2.legend(fontsize=7, markerscale=3)

ax3 = fig.add_subplot(2, 3, 3)
top_n    = min(25, len(feature_cols))
top_shap = mean_shap.head(top_n)
def feat_col(f):
    if 'Species' in f: return '#2196F3'
    if 'R0'      in f: return '#4CAF50'
    if 'active'  in f: return '#FF9800'
    return '#9C27B0'
cols = [feat_col(f) for f in top_shap.index]
ax3.barh(range(top_n), top_shap.values[::-1], color=cols[::-1])
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP|'); ax3.set_title(f'Top {top_n} Features (SHAP)')
ax3.legend(handles=[
    Patch(fc='#2196F3', label='Species'),
    Patch(fc='#4CAF50', label='R0'),
    Patch(fc='#FF9800', label='Active'),
    Patch(fc='#9C27B0', label='Structural'),
], loc='lower right', fontsize=7)

ax4 = fig.add_subplot(2, 3, 4)
divs   = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level']==d]['y_true'],
                   results_df[results_df['diversity_level']==d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level'); ax4.set_ylabel('R²')
ax4.set_title('R² by Diversity Level'); ax4.set_ylim(0, 1.05); ax4.legend(fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
sizes  = sorted(results_df['community_size'].unique())
r2_sz  = [r2_score(results_df[results_df['community_size']==n]['y_true'],
                   results_df[results_df['community_size']==n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N'); ax5.set_ylabel('R²')
ax5.set_title('R² by Community Size'); ax5.set_ylim(0, 1.05); ax5.legend(fontsize=8)

ax6 = fig.add_subplot(2, 3, 6)
# Group SHAP totals dynamically based on whichever features are present
groups, values, colours = [], [], []
sp_s = mean_shap[[f for f in feature_cols if 'Species' in f]].sum()
if sp_s > 0:
    groups.append('Species'); values.append(sp_s); colours.append('#2196F3')
r0_s = mean_shap[r0_cols].sum()
groups.append('R0'); values.append(r0_s); colours.append('#4CAF50')
ac_s = mean_shap[active_cols].sum()
groups.append('Active'); values.append(ac_s); colours.append('#FF9800')
st_s = mean_shap[[f for f in feature_cols if f in ['community_size','diversity_level']]].sum()
if st_s > 0:
    groups.append('Structural'); values.append(st_s); colours.append('#9C27B0')
ax6.pie(values, labels=groups, colors=colours,
        autopct='%.1f%%', startangle=90, textprops={'fontsize': 10})
ax6.set_title('SHAP by Feature Group')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(output_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nPlot saved: {output_name}")


Plot saved: lgbm_v2_results_02.png


In [ ]:
# save outputs

model_save  = f"lgbm_v{VERSION}_model.txt"
pred_save   = f"lgbm_v{VERSION}_predictions.csv"

# LightGBM saves in its own text format (not JSON like XGBoost).
# Reload with: m = lgb.Booster(model_file='lgbm_v1_model.txt')
model_final.booster_.save_model(model_save)

pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'residual':        y_test.values - yp_final,
    'diversity_level': results_df['diversity_level'].values,
    'community_size':  results_df['community_size'].values,
}).to_csv(pred_save, index=False)

print(f"Model saved:       {model_save}")
print(f"Predictions saved: {pred_save}")

In [14]:
# summary

print(f"\n{'='*60}")
print(f"SUMMARY — LightGBM VERSION {VERSION}: {version_label}")
print(f"{'='*60}")
print(f"Features:  {len(feature_cols)}")
print(f"R²:        {r2_final:.4f}")
print(f"MAE:       {mae_final:.4f}")
print(f"RMSE:      {rmse_final:.4f}")
print(f"Trees:     {model_final.best_iteration_}")
print(f"\nTop 5 features:")
for feat, val in mean_shap.head(5).items():
    print(f"  {feat:<25} {val:.5f}")


SUMMARY — LightGBM VERSION 2: Nutrients + size (no species)
Features:  42
R²:        0.8624
MAE:       0.0787
RMSE:      0.1404
Trees:     178

Top 5 features:
  R0_c4                     0.08597
  community_size            0.05980
  diversity_level           0.03790
  R0_c11                    0.03304
  R0_c1                     0.03130
